In [1]:
# import from 'glacier_space' repository
import sys

path_repo_glacier_space = '/Users/geoalxx/Python/glacier_space/'
sys.path.append(path_repo_glacier_space)

from scripts.ModelDataEvaluator import ModelDataEvaluator as MDE
from useful_functions import format_datetime64, get_daterange_str

In [2]:
import pandas as pd
import xarray as xr
import numpy as np
from datetime import datetime

data_dir = '../data_in/'
data_out_dir = '../data_out/'

In [3]:
cfg_file_qc = path_repo_glacier_space + 'config/MDE_config_H3_UAV_iMet_generic.ini'
cfg_file_fw = path_repo_glacier_space + 'config/MDE_config_H3_FixedWing_generic.ini'

def get_MDE_for_qc_config(loc, from_date, to_date, movement='up', skipmeters=0):
    cell_id = df_loc[df_loc['New Name'] == loc]['cell_id'].values[0]
    alt = df_loc[df_loc['New Name'] == loc]['Alt'].values[0]

    # update URLs
    icon_file = f'{icon_exp}_LES_30min_51m_ml_{cell_id}.nc'
    uav_path = f'/Users/geoalxx/HEFEX III/Airdata/l4/{loc}/'

    MDE.update_config(cfg_file_qc, 'DIRECTORIES', 'uav_imet', uav_path)
    MDE.update_config(cfg_file_qc, 'FILES', 'icon_data_point', icon_file)
    MDE.update_config(cfg_file_qc, 'ICON', 'from_date', from_date)
    MDE.update_config(cfg_file_qc, 'ICON', 'to_date', to_date)
    MDE.update_config(cfg_file_qc, 'UAV_IMET', 'location', loc)
    MDE.update_config(cfg_file_qc, 'UAV_IMET', 'altitude', str(alt))
    MDE.update_config(cfg_file_qc, 'UAV_IMET', 'movement', movement)
    MDE.update_config(cfg_file_qc, 'UAV_IMET', 'skipmeters', str(skipmeters))

    return MDE(cfg_file_qc, uav_imet=True)


def get_MDE_for_fw_config(loc, from_date, to_date):
    cell_id = df_loc[df_loc['New Name'] == loc]['cell_id'].values[0]
    alt = df_loc[df_loc['New Name'] == loc]['Alt'].values[0]

    # update URLs
    icon_file = f'v370_2030_v2_LES_30min_51m_ml_{cell_id}.nc'

    MDE.update_config(cfg_file_fw, 'FILES', 'icon_data_point', icon_file)
    MDE.update_config(cfg_file_fw, 'ICON', 'from_date', from_date)
    MDE.update_config(cfg_file_fw, 'ICON', 'to_date', to_date)
    MDE.update_config(cfg_file_fw, 'FIXEDWING', 'location', loc)

    return MDE(cfg_file_fw, fixedwing=True)

## Script settings

In [4]:
# selection of variables
sel_var = ['temp','theta', 'qv', 'wspd', 'wdir']
# sel_var = ['temp']

# location settings
# iop, sel_loc = 'IOP1', ['Q255', 'Q267', 'Q267L', 'Q272', 'Q272L', 'Q272R', 'F309']
iop, sel_loc = 'IOP2', ['Q257', 'Q272', 'Q293']

# UAV data settings
max_alt = 500
skipmeters = 3

# ICON settings
icon_nlev = 50     # lowest n ICON model levels
icon_exp = 'v370_2030_v2'

In [5]:
# time settings
if iop == 'IOP1':
    from_date, to_date = '2025-08-07', '2025-08-10'
elif iop == 'IOP2':
    from_date, to_date = '2025-08-18', '2025-08-19'

## Load and prepare location and time dimensions

In [6]:
# load locations
df_loc = pd.read_csv(data_dir + 'H3_Coords_w_cells.csv')
df_loc = df_loc[df_loc['New Name'].isin(sel_loc)]
df_loc

,cell_id,topo,Original Name,New Name,Type,UAV Takeoff,Lat,Lon,Alt,Comment,Alt_DEM_2022
2,34946,2673.014160,Q2,Q257,UAV,x,46.80981,10.78146,2575.0,from iMet,2583.60
6,35704,2798.971191,Q4,Q272,UAV,x,46.80175,10.77113,2720.0,from iMet,2726.18
9,35933,2958.252441,F1,Q293,UAV,x,46.79201,10.75569,2928.0,from iMet,2928.74


In [7]:
# prepare location dimensions
dim_loc = df_loc['New Name']
dim_loc_lat = np.array(df_loc['Lat'])
dim_loc_lon = df_loc['Lon']
dim_loc_cid = df_loc['cell_id'].astype(int)
dim_loc_ele_icon = df_loc['topo']
dim_loc_ele_uav = df_loc['Alt']

In [8]:
# prepare time dimension (full 'to_date' must be included hence +1 day)
dim_time = pd.date_range(pd.to_datetime(from_date,utc=True),pd.to_datetime(to_date,utc=True) + pd.Timedelta(days=1), freq="1h")

## Load ICON and UAV data using `ModelDataEvaluator` framework

In [9]:
# prepare for ICON data
icon_data_info = {}

icon_var_data = {}
for var in sel_var:
    icon_var_data[var] = []

icon_z_mc_data = []  # actual height values in m
dim_height_ml = None    # model level indices

# prepare for UAV data
uav_data_info = {}

uav_var_data = {}
uav_var_data_ml = {}
for var in sel_var:
    uav_var_data[var] = []
    uav_var_data_ml[var] = []

uav_alt_data = []
dim_height = np.arange(0, max_alt, 1)

# process all locations and variables
for loc in dim_loc:
    # initialize MDE for location (individual ICON extract file)
    if loc.startswith('Q'):
        mde = get_MDE_for_qc_config(loc, from_date, to_date, skipmeters=skipmeters)
        loader = mde.uav_imet
    elif loc.startswith('F'):
        mde = get_MDE_for_fw_config(loc, from_date, to_date)
        loader = mde.fw
    else:
        print(f'Location {loc} not recognized. Skipping.')
        continue

    # collect ICON height information
    z_mc = mde.icon.get_altitude_full_level()
    icon_z_mc_data.append(z_mc[-icon_nlev:])

    # collect UAV height information
    uav_ele_launch = mde.get_altitude()
    uav_alt_tmp = np.arange(uav_ele_launch, uav_ele_launch + max_alt, 1)
    uav_alt_data.append(uav_alt_tmp)

    for var in sel_var:
        #--- ICON
        # get actual ICON data and variable info
        df_icon, info_icon = mde.icon.get_data_3D(var, from_date, to_date)

        # shift timestamp assignment from end of interval to start of interval (to match UAV data)
        # e.g. 10 UTC corresponds to 9:30-10:00 of the ICON model run but the UAV operated between 10:00-10:30 UTC,
        # hence the ICON data should be assigned to 10 UTC
        df_icon.iloc[:, :-1] = df_icon.iloc[:, 1:].values

        # only keep timestamps at full hours (to match UAV data)
        full_hr_idx = df_icon.columns[df_icon.columns.minute == 0]
        df_icon = df_icon[full_hr_idx]

        # collect ICON data and info
        icon_var_data[var].append(df_icon[-icon_nlev:])
        icon_data_info[var] = info_icon

        # initialize ICON model level index as dimension
        if dim_height_ml is None:
            dim_height_ml = df_icon.index[-icon_nlev:]

        #--- UAV
        # get UAV data and variable info: original resolution and aggregated to ICON model level
        df_uav, info_uav = loader.get_data(var)
        df_uav_ml,_      = mde.get_aggregated_uav_data(loader, var, df_icon, source_altitude=uav_ele_launch)

        # collect UAV data and info
        df_uav = df_uav.reindex(index=uav_alt_tmp, columns=dim_time) # extent original data to match time dimension and max height
        uav_var_data[var].append(df_uav)
        uav_var_data_ml[var].append(df_uav_ml[-icon_nlev:])
        uav_data_info[var] = info_uav

########################################################################
ModelDataEvaluator:
Loading configuration file: /Users/geoalxx/Python/glacier_space/config/MDE_config_H3_UAV_iMet_generic.ini

------------------------------------------------------------------------
ICONLoader:
Converting half levels to full levels for variable: w
ICON loaded for grid cell: 1; surface altitude: 2674m
Time range: 2025-08-05 00:30:00+00:00 to 2025-08-28 00:00:00+00:00
------------------------------------------------------------------------

------------------------------------------------------------------------
UAViMetLoader:
Loading UAV iMet data from: /Users/geoalxx/HEFEX III/Airdata/l4/Q257//HEFEX3_UAV_Q257_up_*.csv
Location: Q257 (up)
------------------------------------------------------------------------

########################################################################
Adjusting altitude by 99.0m; source altitude: 2575.0m; target altitude: 2674m
Adjusting altitude by 99.0m; source al

## Build datasets

In [10]:
def create_data_array_2D(data, name, height_name, height_dim, var_attrs):
    # get time from dataset
    time_naive = [dt.replace(tzinfo=None) for dt in dim_time]

    # re-arrange shape according to desired dimensions
    data_t = np.array(data).transpose(2, 1, 0)

    # create DataArray
    xr_data = xr.DataArray(
        data_t,
        coords={
            "time": ("time", time_naive, {"long_name": "Timestamp in UTC"}),
            height_name: (height_name, height_dim, {"long_name": "Height index"}),
            "location": ("location", dim_loc, {"long_name": "Location name from field campaign"}),
            "lat": ("location", dim_loc_lat, {"long_name": "Latitude", "units": "degrees_north"}),
            "lon": ("location", dim_loc_lon, {"long_name": "Longitude", "units": "degrees_east"}),
            "ele_icon": ("location", dim_loc_ele_icon, {"long_name": "Elevation on ICON grid", "units": "meters"}),
        },
        dims=["time", height_name, "location"],
        name=name,
        attrs=var_attrs,
    )
    return xr_data

def create_data_array_height_values(data, name, height_name, var_attrs):
    # re-arrange shape according to desired dimensions
    data_t = np.array(data).transpose(1, 0)

    # create DataArray
    xr_data = xr.DataArray(
        data_t,
        dims=[height_name, "location"],
        name=name,
        attrs=var_attrs,
    )
    return xr_data

### ICON

In [11]:
# create new dataset
ds_out = xr.Dataset()

# add 2D variable data to dataset
for var in sel_var:
    icon_data_info[var]['temporal_resolution'] = 'Hourly instantaneous values'
    ds_out[var] = create_data_array_2D(icon_var_data[var], var, 'height_ml', dim_height_ml, icon_data_info[var])

# add model height information (full level center)
_, z_mc_info = mde.icon.data_grid.get_variable_info('z_mc')
ds_out['z_mc'] = create_data_array_height_values(icon_z_mc_data, 'z_mc', 'height_ml', z_mc_info)

# add ICON cell IDs
ds_out.coords['cid'] = ("location", dim_loc_cid, {"long_name": "ICON Cell ID of UAV location"})

# add global attributes
ds_out.attrs = {
    "title": f"Sliced ICON-LES dataset (51m resolution) for HEFEX III UAV locations ({iop})",
    "institution": "Humboldt-Universität zu Berlin",
    "source": "icon-2024.10 (https://gitlab.dkrz.de/icon/icon-model.git)",
    "history": f"Created {datetime.now().strftime('%Y-%m-%d')}",
    "contact": "alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-6761)",
    "campaign": "HEFEX III",
    "experiment_id": f"hefex3/{icon_exp}/exp_R3B15_51m",
    "StartTime": format_datetime64(ds_out.time.values[0]),
    "StopTime": format_datetime64(ds_out.time.values[-1]),
    "comment": f"Experiment based on glacier outlines from OGGM SSP370, year 2030"
}

output_file = data_out_dir + f'icon/HEFEX3__ICON_{icon_exp}_uav_locations_{icon_nlev}ml__1h_{iop}_{get_daterange_str(ds_out)}.nc'
ds_out.to_netcdf(output_file)
print(f'Output file saved as {output_file}.')

Output file saved as ../data_out/icon/HEFEX3__ICON_v370_2030_v2_uav_locations_50ml__1h_IOP2_20250818-20250820.nc.


In [12]:
ds_out

<xarray.Dataset> Size: 149kB
Dimensions:    (time: 49, height_ml: 50, location: 3)
Coordinates:
  * time       (time) datetime64[ns] 392B 2025-08-18 ... 2025-08-20
  * height_ml  (height_ml) int64 400B 101 102 103 104 105 ... 147 148 149 150
  * location   (location) object 24B 'Q257' 'Q272' 'Q293'
    lat        (location) float64 24B 46.81 46.8 46.79
    lon        (location) float64 24B 10.78 10.77 10.76
    ele_icon   (location) float64 24B 2.673e+03 2.799e+03 2.958e+03
    cid        (location) int64 24B 34946 35704 35933
Data variables:
    temp       (time, height_ml, location) float32 29kB 0.8271 0.4301 ... 5.821
    theta      (time, height_ml, location) float32 29kB 311.9 312.3 ... 307.3
    qv         (time, height_ml, location) float32 29kB 2.196 2.088 ... 6.884
    wspd       (time, height_ml, location) float32 29kB 2.462 2.731 ... 1.511
    wdir       (time, height_ml, location) float32 29kB 357.4 357.7 ... 254.7
    z_mc       (height_ml, location) float32 600B 3.924e+03 ... 2.963e+03
Attributes:
    title:          Sliced ICON-LES dataset (51m resolution) for HEFEX III UA...
    institution:    Humboldt-Universität zu Berlin
    source:         icon-2024.10 (https://gitlab.dkrz.de/icon/icon-model.git)
    history:        Created 2026-11-11
    contact:        alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-946...
    campaign:       HEFEX III
    experiment_id:  hefex3/v370_2030_v2/exp_R3B15_51m
    StartTime:      2025-08-18 00:00:00
    StopTime:       2025-08-20 00:00:00
    comment:        Experiment based on glacier outlines from OGGM SSP370, ye...

### Observations from UAV

In [13]:
# create new dataset
ds_out = xr.Dataset()

# add 2D variable data to dataset
for var in sel_var:
    ds_out[f'{var}_ml'] = create_data_array_2D(uav_var_data_ml[var], f'{var}_ml', 'height_ml', dim_height_ml, uav_data_info[var])

    if var == 'wspd' or var == 'wdir':
        uav_data_info[var]['temporal_resolution'] = '5sec averages'
    else:
        uav_data_info[var]['temporal_resolution'] = 'Instantaneous values (1Hz sampling rate)'
    ds_out[var] = create_data_array_2D(uav_var_data[var], var, 'height', dim_height, uav_data_info[var])


# add model height information (full level center) and actual flight altitudes
_, z_mc_info = mde.icon.data_grid.get_variable_info('z_mc')
ds_out['z_mc'] = create_data_array_height_values(icon_z_mc_data, 'z_mc', 'height_ml', z_mc_info)
alt_info={
    "long_name": "flight altitude of UAV",
    "standard_name": "altitude",
    "units": "m",
}
ds_out['altitude'] = create_data_array_height_values(uav_alt_data, 'altitude', 'height', alt_info)

# add global attributes
ds_out.attrs = {
    "title": f"Vertical atmospheric profiles from UAV soundings acquired during {iop} of HEFEX III campaign",
    "institution": "Humboldt-Universität zu Berlin",
    "source": "In situ measurements from DJI Quadcopters with attached iMET XQ-2 sensors; 1 m/s ascent speed",
    "history": f"Created {datetime.now().strftime('%Y-%m-%d')}",
    "contact": "alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-6761)",
    "campaign": "HEFEX III",
    "StartTime": format_datetime64(ds_out.time.values[0]),
    "StopTime": format_datetime64(ds_out.time.values[-1]),
    "comment": f"Original UAV data was additionally aggregated to ICON model levels (*_ml). Data has not been corrected to account for altitude offsets."
}

output_file = data_out_dir + f'obs/HEFEX3__Obs_UAV_profiles_L2__1h_{iop}_{get_daterange_str(ds_out)}.nc'
ds_out.to_netcdf(output_file)
print(f'Output file saved as {output_file}.')

Output file saved as ../data_out/obs/HEFEX3__Obs_UAV_profiles_L2__1h_IOP2_20250818-20250820.nc.


In [14]:
ds_out

<xarray.Dataset> Size: 3MB
Dimensions:    (time: 49, height_ml: 50, location: 3, height: 500)
Coordinates:
  * time       (time) datetime64[ns] 392B 2025-08-18 ... 2025-08-20
  * height_ml  (height_ml) int64 400B 101 102 103 104 105 ... 147 148 149 150
  * location   (location) object 24B 'Q257' 'Q272' 'Q293'
    lat        (location) float64 24B 46.81 46.8 46.79
    lon        (location) float64 24B 10.78 10.77 10.76
    ele_icon   (location) float64 24B 2.673e+03 2.799e+03 2.958e+03
  * height     (height) int64 4kB 0 1 2 3 4 5 6 ... 493 494 495 496 497 498 499
Data variables:
    temp_ml    (time, height_ml, location) float64 59kB nan nan nan ... nan nan
    temp       (time, height, location) float64 588kB nan nan nan ... nan nan
    theta_ml   (time, height_ml, location) float64 59kB nan nan nan ... nan nan
    theta      (time, height, location) float64 588kB nan nan nan ... nan nan
    qv_ml      (time, height_ml, location) float64 59kB nan nan nan ... nan nan
    qv         (time, height, location) float64 588kB nan nan nan ... nan nan
    wspd_ml    (time, height_ml, location) float64 59kB nan nan nan ... nan nan
    wspd       (time, height, location) float64 588kB nan nan nan ... nan nan
    wdir_ml    (time, height_ml, location) float64 59kB nan nan nan ... nan nan
    wdir       (time, height, location) float64 588kB nan nan nan ... nan nan
    z_mc       (height_ml, location) float32 600B 3.924e+03 ... 2.963e+03
    altitude   (height, location) float64 12kB 2.575e+03 2.72e+03 ... 3.427e+03
Attributes:
    title:        Vertical atmospheric profiles from UAV soundings acquired d...
    institution:  Humboldt-Universität zu Berlin
    source:       In situ measurements from DJI Quadcopters with attached iME...
    history:      Created 2026-11-11
    contact:      alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-...
    campaign:     HEFEX III
    StartTime:    2025-08-18 00:00:00
    StopTime:     2025-08-20 00:00:00
    comment:      Original UAV data was additionally aggregated to ICON model...

In [15]:
ds_out.close()